# 3D Drone Gate Racing — FINAL v10 — DIRECT vs SMOOTH

Main experiment: **Direct SAC vs Smooth Curriculum SAC**.

- Same SAC implementation and hyperparameters.
- Same final task: 6 gates, final radius 0.60 m.
- Same training budget and training seeds.
- Same validation and unseen-test tracks.
- The only experimental difference is the task-difficulty schedule:
  - **Direct:** full final task from the first transition.
  - **Smooth:** always 6 gates, with geometry and gate precision increased continuously during training.
- Discrete Curriculum training is disabled.
- Completed agents are released from GPU memory after each run.
- Logs and checkpoints are persisted after every completed method/seed.


## 1. Import e configurazione

In [ ]:
import os
import gc
import json
import math
import copy
import random

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", DEVICE)

if torch.cuda.is_available():
    torch.set_float32_matmul_precision(
        "high"
    )


# ------------------------------------------------------------
# DIRECT VS SMOOTH EXPERIMENT
# ------------------------------------------------------------

TRAIN_SEEDS = [0, 1, 2]

TOTAL_STEPS_PER_METHOD_PER_SEED = 60_000

START_RANDOM_STEPS = 2_500

BATCH_SIZE = 256
UPDATE_EVERY = 2

HIDDEN = 128
SAC_ALPHA = 0.10

EVAL_EVERY = 5_000
VALIDATION_EPISODES = 5

# Procedural track pools
TRAIN_TRACK_SEEDS = list(
    range(0, 64)
)

VALIDATION_TRACK_SEEDS = list(
    range(5_000, 5_005)
)

TEST_TRACK_SEEDS = list(
    range(10_000, 10_020)
)

CHECKPOINT_DIR = (
    "drone_3d_final_checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print()
print("Training seeds:", TRAIN_SEEDS)
print(
    "Steps per method/seed:",
    TOTAL_STEPS_PER_METHOD_PER_SEED
)
print(
    "Train track pool:",
    len(TRAIN_TRACK_SEEDS)
)
print(
    "Validation tracks:",
    len(VALIDATION_TRACK_SEEDS)
)
print(
    "Final unseen test tracks:",
    len(TEST_TRACK_SEEDS)
)


## 2. Riproducibilità

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


seed_everything(0)

print("Global seed initialized.")


## 3. Geometria 3D e body frame

In [ ]:
def wrap_angle(angle):
    return (
        angle + math.pi
    ) % (
        2.0 * math.pi
    ) - math.pi


def rotation_matrix_zyx(
    roll,
    pitch,
    yaw,
):
    cr = math.cos(roll)
    sr = math.sin(roll)

    cp = math.cos(pitch)
    sp = math.sin(pitch)

    cy = math.cos(yaw)
    sy = math.sin(yaw)

    return np.array([
        [
            cy * cp,
            cy * sp * sr - sy * cr,
            cy * sp * cr + sy * sr,
        ],
        [
            sy * cp,
            sy * sp * sr + cy * cr,
            sy * sp * cr - cy * sr,
        ],
        [
            -sp,
            cp * sr,
            cp * cr,
        ],
    ], dtype=np.float32)


def normalize_vector(v):
    v = np.asarray(
        v,
        dtype=np.float32
    )

    norm = float(
        np.linalg.norm(v)
    )

    if norm < 1e-8:
        return np.array(
            [1.0, 0.0, 0.0],
            dtype=np.float32
        )

    return (
        v / norm
    ).astype(np.float32)


## 4. Generazione procedurale delle piste

Ogni pista ha 6 gate, generati con un heading (direzione nel piano xy) che evolve gate dopo gate invece di avanzare sempre lungo x:

- ad ogni segmento la direzione sterza di un angolo casuale (fino a ±12°);
- con probabilita' ~13% lo sterzo e' invece un tornante netto (95°–145°), che costringe il drone a tornare indietro rispetto alla direzione di partenza — con un segmento piu' lungo per dargli spazio fisico per eseguire la manovra;
- l'heading di partenza e' fisso, quindi anche il primo gate segue la stessa distribuzione degli altri (non puo' comparire a 360° a caso rispetto all'orientamento iniziale del drone);
- altezza del gate variabile in modo indipendente.

**Calibrazione**: questi parametri (12°, 13%, 95°–145°, segmento piu' lungo sui tornanti) sono stati scelti con un pilota euristico non addestrato (un semplice controllore proporzionale verso il prossimo gate), confrontando crash rate e gate raggiunti contro la vecchia generazione sempre-in-avanti. Con questi valori circa il 20% delle piste richiede una vera inversione di rotta, mantenendo prestazioni del pilota euristico pari o migliori di quelle sulla vecchia generazione — quindi virate reali ma fisicamente eseguibili entro i limiti di rollio/beccheggio del drone (±0.45 rad ≈ 26°). Una prima versione con sterzate fino a ±55° e tornanti al 30% (90°–150°, senza spazio extra) si e' rivelata troppo dura: nei log di training il crash rate scendeva a zero ma il success rate restava a 0.00 con miss rate 60–90%. Un controllo diagnostico ha confermato che i missed gate arrivavano tutti dall'attraversamento fuori raggio (non da un bug nella logica del piano di attraversamento) — la geometria era semplicemente troppo stretta per essere eseguita con precisione.

La normale di ogni gate viene calcolata dalla direzione locale della pista.

Il drone quindi deve attraversare un vero **piano 3D orientato**, non semplicemente superare una coordinata `x`.


In [ ]:
def generate_track(
    track_seed,
    n_gates=6,
    heading_range_deg=10.0,
    hairpin_prob=0.06,
    hairpin_range_deg=(80.0, 120.0),
    seg_normal=(1.00, 1.40),
    seg_hairpin=(2.50, 3.00),
):
    """Genera una pista come sequenza di segmenti con un heading (in
    xy) che evolve gate dopo gate, invece di avanzare sempre lungo +x.

    Ogni segmento sterza di un angolo casuale in
    [-heading_range_deg, +heading_range_deg]; con probabilita'
    `hairpin_prob` lo sterzo e' invece un vero tornante (in
    `hairpin_range_deg`, con segno casuale), a cui viene assegnato un
    segmento piu' lungo (`seg_hairpin`) per dare spazio fisico alla
    manovra. L'heading di partenza e' fisso (0.0): anche il primo
    gate segue la stessa distribuzione degli altri, invece di poter
    saltare a una direzione arbitraria su 360 gradi.

    In v10 la distribuzione finale e' leggermente meno estrema della
    v9: mantiene curve 3D e hairpin occasionali, ma riduce la quota di
    episodi dominati da inversioni molto aggressive. La stessa
    distribuzione finale viene usata da Direct, validation/test e dal
    punto finale dello Smooth Curriculum.
    """

    rng = np.random.default_rng(
        int(track_seed)
    )

    heading = 0.0

    current = np.array(
        [0.0, 0.0, 1.0],
        dtype=np.float32
    )

    positions = []

    for i in range(
        n_gates
    ):
        is_hairpin = (
            rng.uniform()
            < hairpin_prob
        )

        if is_hairpin:
            turn_deg = (
                rng.uniform(
                    *hairpin_range_deg
                )
                * (
                    1.0
                    if rng.uniform() < 0.5
                    else -1.0
                )
            )

            segment_length = rng.uniform(
                *seg_hairpin
            )

        else:
            turn_deg = rng.uniform(
                -heading_range_deg,
                heading_range_deg,
            )

            segment_length = rng.uniform(
                *seg_normal
            )

        heading = (
            heading
            + np.deg2rad(turn_deg)
        )

        dx = segment_length * np.cos(heading)
        dy = segment_length * np.sin(heading)

        dz = rng.uniform(
            -0.20,
            0.20
        )

        current = current + np.array(
            [dx, dy, dz],
            dtype=np.float32
        )

        current[2] = np.clip(
            current[2],
            0.72,
            1.80
        )

        positions.append(
            current.copy()
        )

    positions = np.array(
        positions,
        dtype=np.float32
    )

    normals = []

    start = np.array(
        [0.0, 0.0, 1.0],
        dtype=np.float32
    )

    for i in range(
        n_gates
    ):
        if i == 0:
            tangent = (
                positions[0]
                - start
            )

        elif i == (
            n_gates - 1
        ):
            tangent = (
                positions[i]
                - positions[i - 1]
            )

        else:
            tangent = (
                positions[i + 1]
                - positions[i - 1]
            )

        normals.append(
            normalize_vector(
                tangent
            )
        )

    normals = np.array(
        normals,
        dtype=np.float32
    )

    return {
        "positions": positions,
        "normals": normals,
        "seed": int(track_seed),
    }


# Visual sanity check
example_track = generate_track(
    0
)

print(
    "Track positions shape:",
    example_track[
        "positions"
    ].shape
)

print(
    "Track normals shape:",
    example_track[
        "normals"
    ].shape
)


In [ ]:
def plot_track_3d(
    track,
    gate_radius=0.60,
    title="Procedural track",
):
    fig = plt.figure(
        figsize=(10, 7)
    )

    ax = fig.add_subplot(
        111,
        projection="3d"
    )

    positions = track[
        "positions"
    ]

    normals = track[
        "normals"
    ]

    ax.plot(
        positions[:, 0],
        positions[:, 1],
        positions[:, 2],
        linestyle="--",
        marker="o",
        label="gate centers"
    )

    # Draw local gate axes approximately.
    for i, (
        center,
        normal
    ) in enumerate(
        zip(
            positions,
            normals
        )
    ):
        # Find one vector perpendicular to normal.
        up = np.array(
            [0.0, 0.0, 1.0],
            dtype=np.float32
        )

        side = np.cross(
            normal,
            up
        )

        if (
            np.linalg.norm(
                side
            )
            < 1e-4
        ):
            side = np.array(
                [0.0, 1.0, 0.0],
                dtype=np.float32
            )

        side = normalize_vector(
            side
        )

        vertical = normalize_vector(
            np.cross(
                side,
                normal
            )
        )

        theta = np.linspace(
            0.0,
            2.0 * np.pi,
            80
        )

        circle = (
            center[None, :]
            + gate_radius
            * np.cos(theta)[:, None]
            * side[None, :]
            + gate_radius
            * np.sin(theta)[:, None]
            * vertical[None, :]
        )

        ax.plot(
            circle[:, 0],
            circle[:, 1],
            circle[:, 2]
        )

        ax.text(
            center[0],
            center[1],
            center[2] + gate_radius + 0.08,
            str(i + 1)
        )

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title(title)

    ax.legend()

    plt.show()


plot_track_3d(
    example_track,
    title="Example procedural training track"
)


## 5. 3D Drone Environment

La dinamica è un modello rigid-body 3D semplificato con inner-loop attitude control.

La policy produce:

\[
a=[\phi_d,\theta_d,r_d,\Delta T]
\]

ovvero:

- roll desiderato;
- pitch desiderato;
- yaw-rate desiderato;
- variazione di thrust rispetto all'hover.

### Piccola variabilità fisica

Per evitare una simulazione completamente deterministica, ogni episodio introduce una piccola variabilità **identica per Direct e Curriculum**:

- massa circa ±5%;
- efficienza motori circa ±3%;
- vento debole;
- piccoli errori iniziali.

Questa NON è la variabile sperimentale.


In [ ]:
class DroneGateRacing3DEnv:
    def __init__(
        self,
        mode="direct",
        seed=0,
        dt=0.035,
        max_steps=700,
        train_track_seeds=None,
    ):
        assert mode in {
            "direct",
            "curriculum",
            "evaluation",
        }

        self.mode = mode

        self.dt = float(dt)
        self.max_steps = int(
            max_steps
        )

        self.g = 9.81

        self.m0 = 1.0

        self.I0 = np.array(
            [
                0.022,
                0.025,
                0.040
            ],
            dtype=np.float32
        )

        self.linear_drag = np.array(
            [
                0.10,
                0.10,
                0.13
            ],
            dtype=np.float32
        )

        self.angular_drag = np.array(
            [
                0.06,
                0.06,
                0.08
            ],
            dtype=np.float32
        )

        self.max_roll = 0.45
        self.max_pitch = 0.45

        self.max_yaw_rate = 1.20

        self.max_torque = np.array(
            [
                0.11,
                0.11,
                0.08
            ],
            dtype=np.float32
        )

        self.rng = np.random.default_rng(
            seed
        )

        self.train_track_seeds = (
            list(
                TRAIN_TRACK_SEEDS
            )
            if train_track_seeds
            is None
            else list(
                train_track_seeds
            )
        )

        self.training_progress = (
            1.0
            if mode
            in {
                "direct",
                "evaluation"
            }
            else 0.0
        )

        self.state = np.zeros(
            12,
            dtype=np.float32
        )

        self.track = generate_track(
            self.train_track_seeds[
                0
            ]
        )

        self.xy_min = np.array(
            [-2.0, -2.0],
            dtype=np.float32
        )

        self.xy_max = np.array(
            [2.0, 2.0],
            dtype=np.float32
        )

        self.next_gate = 0
        self.steps = 0

        self.mass = self.m0
        self.inertia = (
            self.I0.copy()
        )

        self.motor_eff = 1.0

        self.wind = np.zeros(
            3,
            dtype=np.float32
        )

        self.obs_dim = 22
        self.action_dim = 4

        self.prev_gate_distance = 0.0
        self.prev_radial_error = 0.0
        self.prev_signed_distance = 0.0

        self.reset(
            seed=seed
        )

    # --------------------------------------------------------
    # TASK CURRICULUM
    # --------------------------------------------------------

    def set_training_progress(
        self,
        p,
    ):
        if self.mode == "curriculum":
            self.training_progress = float(
                np.clip(
                    p,
                    0.0,
                    1.0
                )
            )
        else:
            self.training_progress = 1.0

    def task_parameters(
        self,
    ):
        if self.mode in {
            "direct",
            "evaluation",
        }:
            return 6, 0.60

        p = self.training_progress

        # v10 discrete curriculum:
        # 0-20%   -> 2 gates, r=.95
        # 20-40%  -> 3 gates, r=.85
        # 40-60%  -> 4 gates, r=.75
        # 60-75%  -> 5 gates, r=.68
        # 75-100% -> final task: 6 gates, r=.60

        if p < 0.20:
            return 2, 0.95

        if p < 0.40:
            return 3, 0.85

        if p < 0.60:
            return 4, 0.75

        if p < 0.75:
            return 5, 0.68

        return 6, 0.60

    # --------------------------------------------------------
    # SMALL FIXED PHYSICAL NOISE
    # --------------------------------------------------------

    def _sample_episode_physics(
        self,
    ):
        self.mass = (
            self.m0
            * self.rng.uniform(
                0.95,
                1.05
            )
        )

        inertia_scale = (
            self.rng.uniform(
                0.95,
                1.05,
                size=3
            )
        )

        self.inertia = (
            self.I0
            * inertia_scale
        ).astype(
            np.float32
        )

        self.motor_eff = float(
            self.rng.uniform(
                0.97,
                1.00
            )
        )

        self.wind = self.rng.uniform(
            low=np.array(
                [
                    -0.05,
                    -0.10,
                    -0.05
                ]
            ),
            high=np.array(
                [
                    0.05,
                    0.10,
                    0.05
                ]
            ),
        ).astype(np.float32)

    # --------------------------------------------------------
    # TRACK / GATE HELPERS
    # --------------------------------------------------------

    def _active_gate_count(
        self,
    ):
        n, _ = self.task_parameters()

        return min(
            n,
            len(
                self.track[
                    "positions"
                ]
            )
        )

    def _current_gate_index(
        self,
    ):
        return min(
            self.next_gate,
            self._active_gate_count()
            - 1
        )

    def _gate_pose(
        self,
        gate_index,
    ):
        gate_index = int(
            np.clip(
                gate_index,
                0,
                len(
                    self.track[
                        "positions"
                    ]
                )
                - 1
            )
        )

        return (
            self.track[
                "positions"
            ][gate_index],
            self.track[
                "normals"
            ][gate_index],
        )

    def _signed_gate_distance(
        self,
        position=None,
        gate_index=None,
    ):
        if position is None:
            position = self.state[
                0:3
            ]

        if gate_index is None:
            gate_index = (
                self._current_gate_index()
            )

        center, normal = (
            self._gate_pose(
                gate_index
            )
        )

        return float(
            np.dot(
                position - center,
                normal
            )
        )

    def _radial_gate_error(
        self,
        position=None,
        gate_index=None,
    ):
        if position is None:
            position = self.state[
                0:3
            ]

        if gate_index is None:
            gate_index = (
                self._current_gate_index()
            )

        center, normal = (
            self._gate_pose(
                gate_index
            )
        )

        rel = position - center

        signed = np.dot(
            rel,
            normal
        )

        planar = (
            rel
            - signed
            * normal
        )

        return float(
            np.linalg.norm(
                planar
            )
        )

    def _gate_distance(
        self,
    ):
        center, _ = (
            self._gate_pose(
                self._current_gate_index()
            )
        )

        return float(
            np.linalg.norm(
                center
                - self.state[0:3]
            )
        )

    # --------------------------------------------------------
    # BODY-FRAME OBSERVATION
    # --------------------------------------------------------

    def _gate_features_body(
        self,
        gate_index,
    ):
        pos = self.state[
            0:3
        ]

        roll, pitch, yaw = (
            self.state[6:9]
        )

        R = rotation_matrix_zyx(
            float(roll),
            float(pitch),
            float(yaw)
        )

        center, normal_world = (
            self._gate_pose(
                gate_index
            )
        )

        rel_world = (
            center - pos
        )

        rel_body = (
            R.T
            @ rel_world
        )

        normal_body = (
            R.T
            @ normal_world
        )

        features = np.concatenate([
            rel_body
            / np.array(
                [
                    2.0,
                    1.5,
                    1.5
                ],
                dtype=np.float32
            ),
            normal_body,
        ]).astype(
            np.float32
        )

        return features

    def _observation(
        self,
    ):
        roll, pitch, yaw = (
            self.state[6:9]
        )

        R = rotation_matrix_zyx(
            float(roll),
            float(pitch),
            float(yaw)
        )

        vel_world = self.state[
            3:6
        ]

        vel_body = (
            R.T
            @ vel_world
        )

        # Gravity direction in body frame acts as a compact
        # attitude cue without absolute yaw dependence.
        gravity_world_unit = np.array(
            [
                0.0,
                0.0,
                -1.0
            ],
            dtype=np.float32
        )

        gravity_body = (
            R.T
            @ gravity_world_unit
        )

        rates = self.state[
            9:12
        ]

        active_count = (
            self._active_gate_count()
        )

        g1 = self._current_gate_index()

        g2 = min(
            g1 + 1,
            active_count - 1
        )

        gate1_features = (
            self._gate_features_body(
                g1
            )
        )

        gate2_features = (
            self._gate_features_body(
                g2
            )
        )

        progress_fraction = (
            self.next_gate
            / max(
                1,
                active_count
            )
        )

        obs = np.concatenate([
            vel_body
            / np.array(
                [
                    4.0,
                    3.0,
                    3.0
                ],
                dtype=np.float32
            ),

            gravity_body,

            rates
            / np.array(
                [
                    4.0,
                    4.0,
                    4.0
                ],
                dtype=np.float32
            ),

            gate1_features,

            gate2_features,

            np.array(
                [
                    progress_fraction
                ],
                dtype=np.float32
            ),
        ]).astype(
            np.float32
        )

        return np.clip(
            obs,
            -5.0,
            5.0
        )

    # --------------------------------------------------------
    # RESET
    # --------------------------------------------------------

    def _update_track_bounds(
        self,
    ):
        """Ricalcola i confini di sicurezza in xy in base
        all'ingombro reale della pista appena generata, invece di
        usare bound fissi (validi solo per un percorso monotono in
        avanti). Include sempre l'origine, dato che il drone parte
        li'."""

        xy_margin = 1.20

        positions_xy = self.track[
            "positions"
        ][:, :2]

        origin_xy = np.zeros(
            2,
            dtype=np.float32
        )

        all_xy = np.vstack([
            positions_xy,
            origin_xy[None, :],
        ])

        self.xy_min = (
            all_xy.min(axis=0)
            - xy_margin
        ).astype(np.float32)

        self.xy_max = (
            all_xy.max(axis=0)
            + xy_margin
        ).astype(np.float32)

    def reset(
        self,
        seed=None,
        track_seed=None,
        deterministic=False,
    ):
        if seed is not None:
            self.rng = (
                np.random.default_rng(
                    int(seed)
                )
            )

        if track_seed is None:
            track_seed = int(
                self.rng.choice(
                    self.train_track_seeds
                )
            )

        self.track = generate_track(
            int(track_seed)
        )

        self._update_track_bounds()

        self.steps = 0
        self.next_gate = 0

        if deterministic:
            self.mass = self.m0
            self.inertia = (
                self.I0.copy()
            )
            self.motor_eff = 1.0
            self.wind[:] = 0.0

            position = np.array(
                [
                    0.0,
                    0.0,
                    1.0
                ],
                dtype=np.float32
            )

            velocity = np.zeros(
                3,
                dtype=np.float32
            )

            attitude = np.zeros(
                3,
                dtype=np.float32
            )

            rates = np.zeros(
                3,
                dtype=np.float32
            )

        else:
            self._sample_episode_physics()

            position = (
                np.array(
                    [
                        0.0,
                        0.0,
                        1.0
                    ],
                    dtype=np.float32
                )
                +
                self.rng.uniform(
                    -0.035,
                    0.035,
                    size=3
                ).astype(
                    np.float32
                )
            )

            velocity = (
                self.rng.uniform(
                    -0.05,
                    0.05,
                    size=3
                ).astype(
                    np.float32
                )
            )

            attitude = (
                self.rng.uniform(
                    -0.025,
                    0.025,
                    size=3
                ).astype(
                    np.float32
                )
            )

            rates = (
                self.rng.uniform(
                    -0.04,
                    0.04,
                    size=3
                ).astype(
                    np.float32
                )
            )

        self.state[:] = np.concatenate([
            position,
            velocity,
            attitude,
            rates
        ]).astype(
            np.float32
        )

        self.prev_gate_distance = (
            self._gate_distance()
        )

        self.prev_radial_error = (
            self._radial_gate_error()
        )

        self.prev_signed_distance = (
            self._signed_gate_distance()
        )

        return self._observation()

    # --------------------------------------------------------
    # DYNAMICS + REWARD
    # --------------------------------------------------------

    def step(
        self,
        action,
    ):
        action = np.asarray(
            action,
            dtype=np.float32
        )

        action = np.clip(
            action,
            -1.0,
            1.0
        )

        pos = self.state[
            0:3
        ].astype(np.float64)

        vel = self.state[
            3:6
        ].astype(np.float64)

        roll, pitch, yaw = map(
            float,
            self.state[6:9]
        )

        omega = self.state[
            9:12
        ].astype(np.float64)

        previous_position = (
            pos.copy()
        )

        # ----------------------------------------------------
        # RL COMMAND
        # ----------------------------------------------------

        desired_roll = (
            self.max_roll
            * float(
                action[0]
            )
        )

        desired_pitch = (
            self.max_pitch
            * float(
                action[1]
            )
        )

        desired_yaw_rate = (
            self.max_yaw_rate
            * float(
                action[2]
            )
        )

        thrust = (
            self.mass
            * self.g
            * (
                1.0
                + 0.35
                * float(
                    action[3]
                )
            )
            * self.motor_eff
        )

        thrust = float(
            np.clip(
                thrust,
                0.30
                * self.mass
                * self.g,

                1.60
                * self.mass
                * self.g
            )
        )

        # ----------------------------------------------------
        # INNER ATTITUDE CONTROLLER
        # ----------------------------------------------------

        attitude_error = np.array(
            [
                desired_roll
                - roll,

                desired_pitch
                - pitch,

                0.0
            ],
            dtype=np.float64
        )

        desired_rates = np.array(
            [
                0.0,
                0.0,
                desired_yaw_rate
            ],
            dtype=np.float64
        )

        kp = np.array(
            [
                0.32,
                0.32,
                0.0
            ],
            dtype=np.float64
        )

        kd = np.array(
            [
                0.075,
                0.075,
                0.055
            ],
            dtype=np.float64
        )

        torque = (
            kp
            * attitude_error
            +
            kd
            * (
                desired_rates
                - omega
            )
        )

        torque = np.clip(
            torque,
            -self.max_torque,
            self.max_torque
        )

        # ----------------------------------------------------
        # TRANSLATIONAL DYNAMICS
        # ----------------------------------------------------

        R = rotation_matrix_zyx(
            roll,
            pitch,
            yaw
        ).astype(
            np.float64
        )

        thrust_world = (
            R
            @ np.array(
                [
                    0.0,
                    0.0,
                    thrust
                ],
                dtype=np.float64
            )
        )

        gust = self.rng.normal(
            0.0,
            0.012,
            size=3
        )

        external_force = (
            self.wind.astype(
                np.float64
            )
            + gust
        )

        gravity = np.array(
            [
                0.0,
                0.0,
                -self.g
            ],
            dtype=np.float64
        )

        acceleration = (
            thrust_world
            / self.mass
            +
            gravity
            +
            external_force
            / self.mass
            -
            self.linear_drag.astype(
                np.float64
            )
            * vel
        )

        # ----------------------------------------------------
        # ROTATIONAL DYNAMICS
        # ----------------------------------------------------

        I = self.inertia.astype(
            np.float64
        )

        gyroscopic = np.cross(
            omega,
            I * omega
        )

        omega_dot = (
            torque
            - gyroscopic
            - self.angular_drag.astype(
                np.float64
            )
            * omega
        ) / I

        # Semi-implicit integration
        vel += (
            acceleration
            * self.dt
        )

        pos += (
            vel
            * self.dt
        )

        omega += (
            omega_dot
            * self.dt
        )

        roll += (
            omega[0]
            * self.dt
        )

        pitch += (
            omega[1]
            * self.dt
        )

        yaw += (
            omega[2]
            * self.dt
        )

        roll = wrap_angle(
            roll
        )

        pitch = wrap_angle(
            pitch
        )

        yaw = wrap_angle(
            yaw
        )

        self.state[:] = np.array(
            [
                pos[0],
                pos[1],
                pos[2],

                vel[0],
                vel[1],
                vel[2],

                roll,
                pitch,
                yaw,

                omega[0],
                omega[1],
                omega[2],
            ],
            dtype=np.float32
        )

        self.steps += 1

        active_count, gate_radius = (
            self.task_parameters()
        )

        # ----------------------------------------------------
        # POTENTIAL-BASED PROGRESS REWARD
        # ----------------------------------------------------

        gate_distance = (
            self._gate_distance()
        )

        progress_reward = (
            7.0
            * (
                self.prev_gate_distance
                - gate_distance
            )
        )

        self.prev_gate_distance = (
            gate_distance
        )

        radial_error = (
            self._radial_gate_error()
        )

        alignment_reward = (
            3.0
            * (
                self.prev_radial_error
                - radial_error
            )
        )

        self.prev_radial_error = (
            radial_error
        )

        reward = (
            progress_reward
            + alignment_reward
        )

        # Time cost
        reward -= 0.012

        # Stabilization costs
        reward -= (
            0.009
            * (
                roll**2
                + pitch**2
            )
        )

        reward -= (
            0.0015
            * float(
                np.dot(
                    omega,
                    omega
                )
            )
        )

        reward -= (
            0.003
            * float(
                np.dot(
                    action,
                    action
                )
            )
        )

        speed = float(
            np.linalg.norm(
                vel
            )
        )

        if speed > 3.2:
            reward -= (
                0.020
                * (
                    speed - 3.2
                )**2
            )

        # ----------------------------------------------------
        # TRUE 3D GATE-PLANE CROSSING
        # ----------------------------------------------------

        gate_index = (
            self._current_gate_index()
        )

        signed_previous = (
            self._signed_gate_distance(
                position=(
                    previous_position
                ),
                gate_index=(
                    gate_index
                ),
            )
        )

        signed_current = (
            self._signed_gate_distance(
                position=pos,
                gate_index=gate_index,
            )
        )

        crossed_plane = bool(
            signed_previous
            < 0.0
            <= signed_current
        )

        gate_reached = False
        missed_gate = False

        if crossed_plane:
            crossing_radial_error = (
                self._radial_gate_error(
                    position=pos,
                    gate_index=gate_index,
                )
            )

            if (
                crossing_radial_error
                <= gate_radius
            ):
                gate_reached = True

                precision = (
                    1.0
                    - crossing_radial_error
                    / gate_radius
                )

                reward += (
                    22.0
                    + 5.0
                    * precision
                )

                self.next_gate += 1

                if (
                    self.next_gate
                    < active_count
                ):
                    self.prev_gate_distance = (
                        self._gate_distance()
                    )

                    self.prev_radial_error = (
                        self._radial_gate_error()
                    )

            else:
                missed_gate = True
                reward -= 15.0

        # Clearly passed the plane without valid gate passage.
        if (
            not missed_gate
            and self.next_gate
            < active_count
        ):
            current_signed = (
                self._signed_gate_distance()
            )

            if current_signed > 0.22:
                missed_gate = True
                reward -= 15.0

        success = bool(
            self.next_gate
            >= active_count
        )

        # ----------------------------------------------------
        # SAFETY
        # ----------------------------------------------------

        crash = bool(
            pos[2] < 0.08
            or pos[2] > 3.0
            or pos[0] < self.xy_min[0]
            or pos[0] > self.xy_max[0]
            or pos[1] < self.xy_min[1]
            or pos[1] > self.xy_max[1]
            or abs(roll) > 1.30
            or abs(pitch) > 1.30
        )

        terminated = bool(
            success
            or crash
            or missed_gate
        )

        truncated = bool(
            self.steps
            >= self.max_steps
            and not terminated
        )

        if success:
            reward += 80.0

        if crash:
            reward -= 25.0

        info = {
            "gates": int(
                self.next_gate
            ),
            "active_gates": int(
                active_count
            ),
            "gate_radius": float(
                gate_radius
            ),
            "success": bool(
                success
            ),
            "crash": bool(
                crash
            ),
            "missed_gate": bool(
                missed_gate
            ),
            "gate_reached": bool(
                gate_reached
            ),
            "radial_error": float(
                0.0
                if success
                else self._radial_gate_error()
            ),
            "track_seed": int(
                self.track[
                    "seed"
                ]
            ),
        }

        return (
            self._observation(),
            float(
                reward
            ),
            terminated,
            truncated,
            info
        )


## 6. Sanity check della perception e della fisica

In [ ]:
env_test = DroneGateRacing3DEnv(
    mode="evaluation",
    seed=123
)

obs = env_test.reset(
    seed=123,
    track_seed=10_001,
    deterministic=True,
)

print(
    "Observation shape:",
    obs.shape
)

print(
    "Expected obs dim:",
    env_test.obs_dim
)

assert obs.shape == (
    env_test.obs_dim,
)

# Verify body-frame gate features change with yaw.
features_yaw0 = (
    env_test._gate_features_body(
        0
    ).copy()
)

env_test.state[8] = (
    np.pi / 2.0
)

features_yaw90 = (
    env_test._gate_features_body(
        0
    ).copy()
)

print(
    "Gate body-frame features at yaw=0:",
    np.round(
        features_yaw0,
        3
    )
)

print(
    "Gate body-frame features at yaw=90deg:",
    np.round(
        features_yaw90,
        3
    )
)

print(
    "✓ Perception is expressed in ego/body frame"
)


In [ ]:
def simulate_action(
    action,
    steps=80,
):
    env = DroneGateRacing3DEnv(
        mode="evaluation",
        seed=321
    )

    env.reset(
        seed=321,
        track_seed=10_002,
        deterministic=True,
    )

    states = [
        env.state.copy()
    ]

    for _ in range(
        steps
    ):
        (
            _,
            _,
            terminated,
            truncated,
            _
        ) = env.step(
            action
        )

        states.append(
            env.state.copy()
        )

        if (
            terminated
            or truncated
        ):
            break

    return np.array(
        states
    )


hover = simulate_action(
    [0.0, 0.0, 0.0, 0.0]
)

forward = simulate_action(
    [0.0, 0.30, 0.0, 0.05]
)

side = simulate_action(
    [0.30, 0.0, 0.0, 0.05]
)

up = simulate_action(
    [0.0, 0.0, 0.0, 0.25]
)


print(
    "Hover Δz:",
    hover[-1, 2]
    - hover[0, 2]
)

print(
    "Pitch Δx:",
    forward[-1, 0]
    - forward[0, 0]
)

print(
    "Roll Δy:",
    side[-1, 1]
    - side[0, 1]
)

print(
    "Thrust Δz:",
    up[-1, 2]
    - up[0, 2]
)


## 7. Replay Buffer

In [ ]:
class ReplayBuffer:
    def __init__(
        self,
        obs_dim,
        action_dim,
        capacity=300_000,
        seed=0,
    ):
        self.capacity = int(
            capacity
        )

        self.obs = np.zeros(
            (
                self.capacity,
                obs_dim
            ),
            dtype=np.float32
        )

        self.actions = np.zeros(
            (
                self.capacity,
                action_dim
            ),
            dtype=np.float32
        )

        self.rewards = np.zeros(
            (
                self.capacity,
                1
            ),
            dtype=np.float32
        )

        self.next_obs = np.zeros(
            (
                self.capacity,
                obs_dim
            ),
            dtype=np.float32
        )

        self.dones = np.zeros(
            (
                self.capacity,
                1
            ),
            dtype=np.float32
        )

        self.ptr = 0
        self.size = 0

        self.rng = (
            np.random.default_rng(
                seed
            )
        )

    def add(
        self,
        obs,
        action,
        reward,
        next_obs,
        done,
    ):
        i = self.ptr

        self.obs[i] = obs
        self.actions[i] = action
        self.rewards[i] = reward
        self.next_obs[i] = (
            next_obs
        )
        self.dones[i] = done

        self.ptr = (
            self.ptr + 1
        ) % self.capacity

        self.size = min(
            self.size + 1,
            self.capacity
        )

    def sample(
        self,
        batch_size,
        device,
    ):
        idx = self.rng.integers(
            0,
            self.size,
            size=batch_size
        )

        return (
            torch.as_tensor(
                self.obs[idx],
                device=device
            ),
            torch.as_tensor(
                self.actions[idx],
                device=device
            ),
            torch.as_tensor(
                self.rewards[idx],
                device=device
            ),
            torch.as_tensor(
                self.next_obs[idx],
                device=device
            ),
            torch.as_tensor(
                self.dones[idx],
                device=device
            ),
        )


## 8. Soft Actor-Critic — puro PyTorch

In [ ]:
def mlp(
    input_dim,
    output_dim,
    hidden=128,
):
    return nn.Sequential(
        nn.Linear(
            input_dim,
            hidden
        ),
        nn.ReLU(),

        nn.Linear(
            hidden,
            hidden
        ),
        nn.ReLU(),

        nn.Linear(
            hidden,
            output_dim
        ),
    )


class Actor(nn.Module):
    def __init__(
        self,
        obs_dim,
        action_dim,
        hidden=128,
    ):
        super().__init__()

        self.net = mlp(
            obs_dim,
            2 * action_dim,
            hidden
        )

    def forward(
        self,
        obs,
    ):
        out = self.net(
            obs
        )

        mean, log_std = out.chunk(
            2,
            dim=-1
        )

        log_std = torch.clamp(
            log_std,
            -5.0,
            1.5
        )

        return (
            mean,
            log_std
        )

    def sample(
        self,
        obs,
        deterministic=False,
    ):
        mean, log_std = self(
            obs
        )

        if deterministic:
            pre_tanh = mean
        else:
            std = (
                log_std.exp()
            )

            pre_tanh = (
                mean
                + std
                * torch.randn_like(
                    mean
                )
            )

        action = torch.tanh(
            pre_tanh
        )

        if deterministic:
            return (
                action,
                None
            )

        std = log_std.exp()

        normal_log_prob = -0.5 * (
            (
                (
                    pre_tanh
                    - mean
                )
                /
                (
                    std
                    + 1e-8
                )
            )**2
            +
            2.0
            * log_std
            +
            math.log(
                2.0
                * math.pi
            )
        )

        log_prob = (
            normal_log_prob.sum(
                dim=-1,
                keepdim=True
            )
        )

        log_prob -= torch.log(
            1.0
            - action.pow(2)
            + 1e-6
        ).sum(
            dim=-1,
            keepdim=True
        )

        return (
            action,
            log_prob
        )


class Critic(nn.Module):
    def __init__(
        self,
        obs_dim,
        action_dim,
        hidden=128,
    ):
        super().__init__()

        self.net = mlp(
            obs_dim
            + action_dim,
            1,
            hidden
        )

    def forward(
        self,
        obs,
        action,
    ):
        return self.net(
            torch.cat(
                [
                    obs,
                    action
                ],
                dim=-1
            )
        )


class SACAgent:
    def __init__(
        self,
        obs_dim,
        action_dim,
        hidden=128,
        lr=3e-4,
        gamma=0.99,
        tau=0.005,
        alpha=0.10,
        device=DEVICE,
    ):
        self.device = device

        self.obs_dim = int(
            obs_dim
        )

        self.action_dim = int(
            action_dim
        )

        self.hidden = int(
            hidden
        )

        self.gamma = float(
            gamma
        )

        self.tau = float(
            tau
        )

        self.alpha = float(
            alpha
        )

        self.actor = Actor(
            obs_dim,
            action_dim,
            hidden
        ).to(device)

        self.q1 = Critic(
            obs_dim,
            action_dim,
            hidden
        ).to(device)

        self.q2 = Critic(
            obs_dim,
            action_dim,
            hidden
        ).to(device)

        self.target_q1 = (
            copy.deepcopy(
                self.q1
            ).to(device)
        )

        self.target_q2 = (
            copy.deepcopy(
                self.q2
            ).to(device)
        )

        self.actor_optimizer = (
            torch.optim.Adam(
                self.actor.parameters(),
                lr=lr
            )
        )

        self.q1_optimizer = (
            torch.optim.Adam(
                self.q1.parameters(),
                lr=lr
            )
        )

        self.q2_optimizer = (
            torch.optim.Adam(
                self.q2.parameters(),
                lr=lr
            )
        )

    @torch.no_grad()
    def act(
        self,
        obs,
        deterministic=False,
    ):
        obs_t = torch.as_tensor(
            obs,
            dtype=torch.float32,
            device=self.device
        ).unsqueeze(0)

        action, _ = (
            self.actor.sample(
                obs_t,
                deterministic=(
                    deterministic
                )
            )
        )

        return (
            action
            .squeeze(0)
            .cpu()
            .numpy()
        )

    def update(
        self,
        batch,
    ):
        (
            obs,
            actions,
            rewards,
            next_obs,
            dones
        ) = batch

        with torch.no_grad():
            (
                next_actions,
                next_log_prob
            ) = self.actor.sample(
                next_obs
            )

            target_q = torch.min(
                self.target_q1(
                    next_obs,
                    next_actions
                ),
                self.target_q2(
                    next_obs,
                    next_actions
                )
            )

            target_q = (
                target_q
                - self.alpha
                * next_log_prob
            )

            y = (
                rewards
                + self.gamma
                * (
                    1.0
                    - dones
                )
                * target_q
            )

        q1_value = self.q1(
            obs,
            actions
        )

        q1_loss = F.mse_loss(
            q1_value,
            y
        )

        self.q1_optimizer.zero_grad(
            set_to_none=True
        )

        q1_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.q1.parameters(),
            10.0
        )

        self.q1_optimizer.step()

        q2_value = self.q2(
            obs,
            actions
        )

        q2_loss = F.mse_loss(
            q2_value,
            y
        )

        self.q2_optimizer.zero_grad(
            set_to_none=True
        )

        q2_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.q2.parameters(),
            10.0
        )

        self.q2_optimizer.step()

        (
            new_actions,
            log_prob
        ) = self.actor.sample(
            obs
        )

        q_pi = torch.min(
            self.q1(
                obs,
                new_actions
            ),
            self.q2(
                obs,
                new_actions
            )
        )

        actor_loss = (
            self.alpha
            * log_prob
            - q_pi
        ).mean()

        self.actor_optimizer.zero_grad(
            set_to_none=True
        )

        actor_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.actor.parameters(),
            10.0
        )

        self.actor_optimizer.step()

        with torch.no_grad():
            for p, tp in zip(
                self.q1.parameters(),
                self.target_q1.parameters()
            ):
                tp.data.mul_(
                    1.0
                    - self.tau
                ).add_(
                    self.tau
                    * p.data
                )

            for p, tp in zip(
                self.q2.parameters(),
                self.target_q2.parameters()
            ):
                tp.data.mul_(
                    1.0
                    - self.tau
                ).add_(
                    self.tau
                    * p.data
                )

        return {
            "q1_loss": float(
                q1_loss.item()
            ),
            "q2_loss": float(
                q2_loss.item()
            ),
            "actor_loss": float(
                actor_loss.item()
            ),
            "alpha": float(
                self.alpha
            ),
        }

    def save(
        self,
        path,
    ):
        torch.save({
            "actor": self.actor.state_dict(),
            "q1": self.q1.state_dict(),
            "q2": self.q2.state_dict(),
            "target_q1": (
                self.target_q1.state_dict()
            ),
            "target_q2": (
                self.target_q2.state_dict()
            ),
            "obs_dim": self.obs_dim,
            "action_dim": self.action_dim,
            "hidden": self.hidden,
            "gamma": self.gamma,
            "tau": self.tau,
            "alpha": self.alpha,
        }, path)

    @classmethod
    def load(
        cls,
        path,
        device=DEVICE,
    ):
        checkpoint = torch.load(
            path,
            map_location=device
        )

        agent = cls(
            obs_dim=checkpoint[
                "obs_dim"
            ],
            action_dim=checkpoint[
                "action_dim"
            ],
            hidden=checkpoint[
                "hidden"
            ],
            gamma=checkpoint[
                "gamma"
            ],
            tau=checkpoint[
                "tau"
            ],
            alpha=checkpoint[
                "alpha"
            ],
            device=device,
        )

        agent.actor.load_state_dict(
            checkpoint[
                "actor"
            ]
        )

        agent.q1.load_state_dict(
            checkpoint[
                "q1"
            ]
        )

        agent.q2.load_state_dict(
            checkpoint[
                "q2"
            ]
        )

        agent.target_q1.load_state_dict(
            checkpoint[
                "target_q1"
            ]
        )

        agent.target_q2.load_state_dict(
            checkpoint[
                "target_q2"
            ]
        )

        return agent


## 9. Smoke test SAC

In [ ]:
smoke_env = (
    DroneGateRacing3DEnv(
        mode="curriculum",
        seed=0
    )
)

smoke_env.set_training_progress(
    0.10
)

smoke_obs = smoke_env.reset(
    seed=0
)

smoke_buffer = ReplayBuffer(
    smoke_env.obs_dim,
    smoke_env.action_dim,
    capacity=3_000,
    seed=0,
)

for _ in range(
    600
):
    action = (
        smoke_env.rng.uniform(
            -1.0,
            1.0,
            size=(
                smoke_env.action_dim
            )
        ).astype(
            np.float32
        )
    )

    (
        next_obs,
        reward,
        terminated,
        truncated,
        info
    ) = smoke_env.step(
        action
    )

    done = bool(
        terminated
        or truncated
    )

    smoke_buffer.add(
        smoke_obs,
        action,
        reward,
        next_obs,
        float(
            done
        )
    )

    smoke_obs = next_obs

    if done:
        smoke_obs = (
            smoke_env.reset()
        )

smoke_agent = SACAgent(
    smoke_env.obs_dim,
    smoke_env.action_dim,
    hidden=HIDDEN,
    alpha=SAC_ALPHA,
)

losses = smoke_agent.update(
    smoke_buffer.sample(
        BATCH_SIZE,
        DEVICE
    )
)

print(
    "One SAC update:",
    losses
)

print(
    "✓ Environment + replay + SAC ready"
)


## 10. Evaluation utility

Durante il training **tutti e tre gli agenti** vengono valutati ogni 5k step sulla **stessa validation set**:

- task finale completa;
- 6 gate;
- raggio 0.60 m;
- stessa distribuzione geometrica finale;
- seed separati dal training.

Questo mantiene il confronto di sample efficiency equo: Curriculum e Smooth possono ricevere task più semplici durante il training, ma vengono sempre misurati sulla **stessa final task usata da Direct**.


In [ ]:
@torch.no_grad()
def evaluate_on_tracks(
    agent,
    track_seeds,
    episodes_per_track=1,
):
    gates = []
    successes = []
    crashes = []
    misses = []
    lengths = []

    for track_seed in (
        track_seeds
    ):
        for repetition in range(
            episodes_per_track
        ):
            env = (
                DroneGateRacing3DEnv(
                    mode="evaluation",
                    seed=(
                        int(track_seed)
                        + 100
                        * repetition
                    )
                )
            )

            obs = env.reset(
                seed=(
                    int(track_seed)
                    + 100
                    * repetition
                ),
                track_seed=(
                    int(track_seed)
                ),
                deterministic=True,
            )

            done = False

            while not done:
                action = agent.act(
                    obs,
                    deterministic=True
                )

                (
                    obs,
                    _,
                    terminated,
                    truncated,
                    info
                ) = env.step(
                    action
                )

                done = bool(
                    terminated
                    or truncated
                )

            gates.append(
                info[
                    "gates"
                ]
            )

            successes.append(
                int(
                    info[
                        "success"
                    ]
                )
            )

            crashes.append(
                int(
                    info[
                        "crash"
                    ]
                )
            )

            misses.append(
                int(
                    info[
                        "missed_gate"
                    ]
                )
            )

            lengths.append(
                env.steps
            )

    return {
        "mean_gates": float(
            np.mean(
                gates
            )
        ),
        "std_gates": float(
            np.std(
                gates
            )
        ),
        "success_rate": float(
            np.mean(
                successes
            )
        ),
        "crash_rate": float(
            np.mean(
                crashes
            )
        ),
        "miss_rate": float(
            np.mean(
                misses
            )
        ),
        "mean_length": float(
            np.mean(
                lengths
            )
        ),
    }


## 10b. Smooth Curriculum — piste a difficolta' continua

Oltre a Direct (task fisso) e Discrete Curriculum (difficolta' a gradini 2→3→4→5→6 gate), introduciamo uno **Smooth Curriculum**. Qui il numero di gate resta sempre 6, mentre raggio dei gate e complessità geometrica cambiano continuamente.

La difficoltà raggiunge 1.0 al 75% del training (45k su 60k): gli ultimi 15k step sono quindi interamente sulla stessa final task usata da Direct e dall'evaluation. A difficulty=1 la distribuzione geometrica coincide con `generate_track()`.


In [ ]:
def generate_smooth_curriculum_track(
    track_seed,
    difficulty,
    n_gates=6,
):
    """
    difficulty = 0:
        quasi rettilinea, sterzate minime, niente tornanti.

    difficulty = 1:
        stessa distribuzione geometrica di generate_track()
        (heading_range_deg=10, hairpin_prob=0.06,
        hairpin_range=80-120deg, stessa distribuzione dei segmenti),
        quindi include le stesse virate e gli stessi occasionali
        tornanti che richiedono di tornare indietro.
    """

    difficulty = float(
        np.clip(
            difficulty,
            0.0,
            1.0,
        )
    )

    rng = np.random.default_rng(
        int(track_seed)
    )

    # --------------------------------------------------------
    # GEOMETRY DIFFICULTY
    # --------------------------------------------------------

    heading_range_deg = (
        2.0
        + 8.0
        * difficulty
    )

    hairpin_prob = (
        0.06
        * difficulty
    )

    hairpin_low = (
        60.0
        + 20.0
        * difficulty
    )

    hairpin_high = (
        80.0
        + 40.0
        * difficulty
    )

    vertical_max = (
        0.03
        + 0.17
        * difficulty
    )

    dx_low = (
        1.20
        - 0.20
        * difficulty
    )

    dx_high = (
        1.20
        + 0.20
        * difficulty
    )

    heading = 0.0

    current = np.array(
        [
            0.0,
            0.0,
            1.0,
        ],
        dtype=np.float32,
    )

    positions = []

    for _ in range(
        n_gates
    ):
        is_hairpin = (
            rng.uniform()
            < hairpin_prob
        )

        if is_hairpin:
            turn_deg = (
                rng.uniform(
                    hairpin_low,
                    hairpin_high,
                )
                * (
                    1.0
                    if rng.uniform() < 0.5
                    else -1.0
                )
            )

            segment_length = rng.uniform(
                2.50,
                3.00,
            )

        else:
            turn_deg = rng.uniform(
                -heading_range_deg,
                heading_range_deg,
            )

            segment_length = rng.uniform(
                dx_low,
                dx_high,
            )

        heading = (
            heading
            + np.deg2rad(turn_deg)
        )

        dx = segment_length * np.cos(heading)
        dy = segment_length * np.sin(heading)

        dz = rng.uniform(
            -vertical_max,
            vertical_max,
        )

        current = (
            current
            + np.array(
                [
                    dx,
                    dy,
                    dz,
                ],
                dtype=np.float32,
            )
        )

        current[2] = np.clip(
            current[2],
            0.72,
            1.80,
        )

        positions.append(
            current.copy()
        )

    positions = np.array(
        positions,
        dtype=np.float32,
    )

    # --------------------------------------------------------
    # GATE NORMALS
    # --------------------------------------------------------

    normals = []

    start = np.array(
        [
            0.0,
            0.0,
            1.0,
        ],
        dtype=np.float32,
    )

    for i in range(
        n_gates
    ):
        if i == 0:

            tangent = (
                positions[0]
                - start
            )

        elif i == (
            n_gates - 1
        ):

            tangent = (
                positions[i]
                - positions[i - 1]
            )

        else:

            tangent = (
                positions[i + 1]
                - positions[i - 1]
            )

        normals.append(
            normalize_vector(
                tangent
            )
        )

    normals = np.array(
        normals,
        dtype=np.float32,
    )

    return {
        "positions": positions,
        "normals": normals,
        "seed": int(
            track_seed
        ),
    }


In [ ]:
class SmoothCurriculumEnv(
    DroneGateRacing3DEnv
):
    def __init__(
        self,
        seed=0,
        dt=0.035,
        max_steps=700,
        train_track_seeds=None,
    ):
        self.requested_progress = 0.0
        self.episode_difficulty = 0.0

        super().__init__(
            mode="evaluation",
            seed=seed,
            dt=dt,
            max_steps=max_steps,
            train_track_seeds=(
                train_track_seeds
            ),
        )

    def set_training_progress(
        self,
        progress,
    ):
        progress = float(
            np.clip(
                progress,
                0.0,
                1.0,
            )
        )

        # ----------------------------------------------------
        # Reach full task at 45k / 60k = 75%
        # ----------------------------------------------------
        #
        # 0k  -> difficulty 0
        # 15k -> ~0.33
        # 30k -> ~0.67
        # 45k -> 1.00
        # 60k -> 1.00
        #
        # Last 15k are entirely on final task.

        self.requested_progress = float(
            min(
                1.0,
                progress / 0.75,
            )
        )

    def task_parameters(
        self,
    ):
        # ALWAYS six gates.
        active_gates = 6

        # Smoothly:
        #
        # r = .95  --->  .60

        gate_radius = (
            0.95
            - 0.35
            * self.episode_difficulty
        )

        return (
            active_gates,
            float(
                gate_radius
            ),
        )

    def reset(
        self,
        seed=None,
        track_seed=None,
        deterministic=False,
    ):
        if seed is not None:
            self.rng = (
                np.random.default_rng(
                    int(seed)
                )
            )

        if track_seed is None:

            track_seed = int(
                self.rng.choice(
                    self.train_track_seeds
                )
            )

        # Freeze difficulty during one episode.
        #
        # It therefore DOES NOT change halfway
        # through a trajectory.

        self.episode_difficulty = float(
            self.requested_progress
        )

        obs = super().reset(
            seed=seed,
            track_seed=track_seed,
            deterministic=deterministic,
        )

        # Replace standard track with
        # difficulty-dependent track.

        self.track = (
            generate_smooth_curriculum_track(
                track_seed=track_seed,
                difficulty=(
                    self.episode_difficulty
                ),
                n_gates=6,
            )
        )

        # Ricalcola anche i confini di sicurezza in xy, dato che la
        # pista sostituita puo' avere un ingombro diverso da quella
        # generata da super().reset().
        self._update_track_bounds()

        # Recompute potentials because
        # track changed.

        self.prev_gate_distance = (
            self._gate_distance()
        )

        self.prev_radial_error = (
            self._radial_gate_error()
        )

        self.prev_signed_distance = (
            self._signed_gate_distance()
        )

        return self._observation()

## 11. Training utility

In [ ]:
def make_env_for_mode(
    mode,
    seed,
):
    """Builds the environment for the two methods under comparison.

    Direct uses the complete final task from the beginning.
    Smooth always uses 6 gates but continuously increases geometric
    difficulty and gate precision until it reaches the same final task.
    """

    if mode == "smooth":
        return SmoothCurriculumEnv(
            seed=seed,
            train_track_seeds=(
                TRAIN_TRACK_SEEDS
            ),
        )

    return DroneGateRacing3DEnv(
        mode=mode,
        seed=seed,
        train_track_seeds=(
            TRAIN_TRACK_SEEDS
        ),
    )


def train_agent(
    mode,
    total_steps,
    seed,
):
    assert mode in {
        "direct",
        "smooth",
    }

    seed_everything(
        seed
    )

    env = make_env_for_mode(
        mode,
        seed,
    )

    agent = SACAgent(
        env.obs_dim,
        env.action_dim,
        hidden=HIDDEN,
        alpha=SAC_ALPHA,
        device=DEVICE,
    )

    # v10: stesso replay-buffer budget per tutti i metodi.
    # In questo modo l'unica variabile sperimentale rilevante e'
    # la distribuzione di difficolta' durante il training.
    buffer_capacity = max(
        100_000,
        total_steps + 5_000
    )

    buffer = ReplayBuffer(
        env.obs_dim,
        env.action_dim,
        capacity=buffer_capacity,
        seed=seed,
    )

    if mode == "smooth":
        env.set_training_progress(
            0.0
        )
    else:
        env.set_training_progress(
            1.0
        )

    obs = env.reset(
        seed=seed
    )

    episode_return = 0.0
    episode_length = 0

    episode_radial = []
    episode_missed = 0

    train_logs = []
    validation_logs = []

    best_validation_gates = (
        -np.inf
    )

    best_checkpoint_path = (
        os.path.join(
            CHECKPOINT_DIR,
            f"best_{mode}_seed{seed}.pt"
        )
    )

    last_losses = {
        "q1_loss": np.nan,
        "q2_loss": np.nan,
        "actor_loss": np.nan,
        "alpha": SAC_ALPHA,
    }

    for step in range(
        1,
        total_steps + 1
    ):
        # -----------------------------------------------
        # TASK DIFFICULTY
        # -----------------------------------------------

        if mode == "smooth":
            env.set_training_progress(
                step
                / total_steps
            )
        else:
            env.set_training_progress(
                1.0
            )

        # -----------------------------------------------
        # ACTION
        # -----------------------------------------------

        if (
            step
            <= START_RANDOM_STEPS
        ):
            action = env.rng.uniform(
                -1.0,
                1.0,
                size=(
                    env.action_dim
                )
            ).astype(
                np.float32
            )

        else:
            action = agent.act(
                obs,
                deterministic=False
            )

        # -----------------------------------------------
        # ENV STEP
        # -----------------------------------------------

        (
            next_obs,
            reward,
            terminated,
            truncated,
            info
        ) = env.step(
            action
        )

        done = bool(
            terminated
            or truncated
        )

        buffer.add(
            obs,
            action,
            reward,
            next_obs,
            float(
                done
            )
        )

        obs = next_obs

        episode_return += reward
        episode_length += 1

        episode_radial.append(
            float(
                info[
                    "radial_error"
                ]
            )
        )

        if info[
            "missed_gate"
        ]:
            episode_missed = 1

        # -----------------------------------------------
        # SAC UPDATE
        # -----------------------------------------------

        if (
            step
            > START_RANDOM_STEPS
            and buffer.size
            >= BATCH_SIZE
            and step
            % UPDATE_EVERY
            == 0
        ):
            last_losses = (
                agent.update(
                    buffer.sample(
                        BATCH_SIZE,
                        DEVICE
                    )
                )
            )

        # -----------------------------------------------
        # EPISODE LOG
        # -----------------------------------------------

        if done:
            train_logs.append({
                "step": int(
                    step
                ),
                "return": float(
                    episode_return
                ),
                "length": int(
                    episode_length
                ),
                "gates": int(
                    info[
                        "gates"
                    ]
                ),
                "active_gates": int(
                    info[
                        "active_gates"
                    ]
                ),
                "gate_radius": float(
                    info[
                        "gate_radius"
                    ]
                ),
                "success": int(
                    info[
                        "success"
                    ]
                ),
                "crash": int(
                    info[
                        "crash"
                    ]
                ),
                "missed_gate": int(
                    episode_missed
                ),
                "mean_radial_error": float(
                    np.mean(
                        episode_radial
                    )
                    if episode_radial
                    else np.nan
                ),
                "track_seed": int(
                    info[
                        "track_seed"
                    ]
                ),
            })

            obs = env.reset()

            episode_return = 0.0
            episode_length = 0
            episode_radial = []
            episode_missed = 0

        # -----------------------------------------------
        # FULL-TASK VALIDATION (identica per tutti i metodi)
        # -----------------------------------------------

        if (
            step
            % EVAL_EVERY
            == 0
        ):
            validation = (
                evaluate_on_tracks(
                    agent,
                    VALIDATION_TRACK_SEEDS,
                    episodes_per_track=1,
                )
            )

            validation_logs.append({
                "step": int(
                    step
                ),
                **validation
            })

            if (
                validation[
                    "mean_gates"
                ]
                > best_validation_gates
            ):
                best_validation_gates = (
                    validation[
                        "mean_gates"
                    ]
                )

                agent.save(
                    best_checkpoint_path
                )

            recent = train_logs[
                -30:
            ]

            if recent:
                train_gates = float(
                    np.mean([
                        row["gates"]
                        for row
                        in recent
                    ])
                )

                train_crash = float(
                    np.mean([
                        row["crash"]
                        for row
                        in recent
                    ])
                )

                train_miss = float(
                    np.mean([
                        row[
                            "missed_gate"
                        ]
                        for row
                        in recent
                    ])
                )

            else:
                train_gates = np.nan
                train_crash = np.nan
                train_miss = np.nan

            active_gates, radius = (
                env.task_parameters()
            )

            print(
                f"{mode:10s} | "
                f"seed {seed} | "
                f"step {step:6d}/{total_steps} | "
                f"train gates {train_gates:.2f} | "
                f"crash {train_crash:.2f} | "
                f"miss {train_miss:.2f} | "
                f"task {active_gates}g/r={radius:.2f} | "
                f"VAL gates {validation['mean_gates']:.2f} | "
                f"VAL success {validation['success_rate']:.2f}"
            )

    final_path = os.path.join(
        CHECKPOINT_DIR,
        f"final_{mode}_seed{seed}.pt"
    )

    agent.save(
        final_path
    )

    return (
        agent,
        train_logs,
        validation_logs,
        best_checkpoint_path,
        final_path,
    )


## 12. FULL TRAINING — Direct SAC vs Smooth Curriculum SAC

Both methods are trained with exactly the same SAC settings and budget. Only the training-task difficulty schedule differs.


In [ ]:
TRAIN_MODES = [
    "direct",
    "smooth",
]

all_train_logs = {
    mode: {}
    for mode in TRAIN_MODES
}

all_validation_logs = {
    mode: {}
    for mode in TRAIN_MODES
}

best_checkpoint_paths = {
    mode: {}
    for mode in TRAIN_MODES
}

final_checkpoint_paths = {
    mode: {}
    for mode in TRAIN_MODES
}


for seed in TRAIN_SEEDS:
    for mode in TRAIN_MODES:
        print()
        print("=" * 90)
        print(
            f"SEED {seed} — {mode.upper()}"
        )
        print("=" * 90)

        (
            agent,
            train_logs,
            validation_logs,
            best_path,
            final_path,
        ) = train_agent(
            mode=mode,
            total_steps=(
                TOTAL_STEPS_PER_METHOD_PER_SEED
            ),
            seed=seed,
        )

        all_train_logs[
            mode
        ][seed] = train_logs

        all_validation_logs[
            mode
        ][seed] = validation_logs

        best_checkpoint_paths[
            mode
        ][seed] = best_path

        final_checkpoint_paths[
            mode
        ][seed] = final_path

        # Save logs immediately after every completed run.
        logs_path = os.path.join(
            CHECKPOINT_DIR,
            f"{mode}_logs_seed{seed}.json"
        )

        with open(
            logs_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                {
                    "mode": mode,
                    "seed": seed,
                    "train_logs": train_logs,
                    "validation_logs": validation_logs,
                    "best_checkpoint": best_path,
                    "final_checkpoint": final_path,
                },
                f,
                indent=2,
            )

        print(
            "Best checkpoint:",
            best_path
        )
        print(
            "Final checkpoint:",
            final_path
        )
        print(
            "Logs:",
            logs_path
        )

        # Kaggle/Colab memory safety:
        # do not keep completed agents resident on GPU.
        del agent
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print()
print(
    "✓ FULL TRAINING COMPLETED (direct vs smooth)"
)


## 13. Sample-efficiency curves — Direct vs Smooth

Validation is always performed on the **same unchanged full final task**, so earlier gains in Smooth directly indicate better sample efficiency rather than an easier validation problem.


In [ ]:
def aggregate_validation_metric(
    mode,
    key,
):
    curves = []

    x = None

    for seed in TRAIN_SEEDS:
        logs = (
            all_validation_logs[
                mode
            ][seed]
        )

        if x is None:
            x = np.array([
                row["step"]
                for row in logs
            ])

        curves.append([
            row[key]
            for row in logs
        ])

    curves = np.array(
        curves,
        dtype=float
    )

    return (
        x,
        np.mean(
            curves,
            axis=0
        ),
        np.std(
            curves,
            axis=0
        ),
    )


# Main comparison: Direct SAC vs Smooth Curriculum SAC.
DEFAULT_VALIDATION_METHODS = [
    ("direct", "Direct SAC"),
    ("smooth", "Smooth Curriculum"),
]


def plot_validation_metric(
    key,
    ylabel,
    methods=None,
):
    if methods is None:
        methods = DEFAULT_VALIDATION_METHODS

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    for mode, label in methods:

        x, mean, std = (
            aggregate_validation_metric(
                mode,
                key
            )
        )

        ax.plot(
            x,
            mean,
            marker="o",
            label=label
        )

        ax.fill_between(
            x,
            mean - std,
            mean + std,
            alpha=0.20
        )

    ax.set_xlabel(
        "Environment transitions"
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        ylabel
        + " on identical full-task validation tracks"
    )

    ax.grid(
        alpha=0.3
    )

    ax.legend()

    plt.show()


plot_validation_metric(
    "mean_gates",
    "Mean gates completed"
)

plot_validation_metric(
    "success_rate",
    "Success rate"
)

plot_validation_metric(
    "miss_rate",
    "Miss rate"
)

plot_validation_metric(
    "crash_rate",
    "Crash rate"
)


## 14. Training diagnostics — Direct vs Smooth


In [ ]:
def binned_training_metric(
    logs,
    key,
    total_steps,
    n_bins=30,
):
    edges = np.linspace(
        0,
        total_steps,
        n_bins + 1
    )

    centers = (
        edges[:-1]
        + edges[1:]
    ) / 2.0

    values = np.full(
        n_bins,
        np.nan,
        dtype=float
    )

    for i in range(
        n_bins
    ):
        bucket = [
            row[key]
            for row in logs
            if (
                edges[i]
                < row["step"]
                <= edges[i + 1]
            )
        ]

        if bucket:
            values[i] = np.mean(
                bucket
            )

    return (
        centers,
        values
    )


def aggregate_training_metric(
    mode,
    key,
):
    curves = []

    x = None

    for seed in TRAIN_SEEDS:
        x, curve = (
            binned_training_metric(
                all_train_logs[
                    mode
                ][seed],
                key,
                TOTAL_STEPS_PER_METHOD_PER_SEED,
                n_bins=30,
            )
        )

        curves.append(
            curve
        )

    curves = np.array(
        curves,
        dtype=float
    )

    return (
        x,
        np.nanmean(
            curves,
            axis=0
        ),
        np.nanstd(
            curves,
            axis=0
        ),
    )


for key, ylabel in [
    (
        "crash",
        "Crash rate"
    ),
    (
        "missed_gate",
        "Missed-gate rate"
    ),
    (
        "mean_radial_error",
        "Mean radial error [m]"
    ),
]:
    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    for mode, label in DEFAULT_VALIDATION_METHODS:
        x, mean, std = (
            aggregate_training_metric(
                mode,
                key
            )
        )

        ax.plot(
            x,
            mean,
            label=label
        )

        ax.fill_between(
            x,
            mean - std,
            mean + std,
            alpha=0.20
        )

    ax.set_xlabel(
        "Environment transitions"
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        ylabel
    )

    ax.grid(
        alpha=0.3
    )

    ax.legend()

    plt.show()


## 15. Best policy selection via validation


In [ ]:
def best_validation_score(
    mode,
    seed,
):
    logs = (
        all_validation_logs[
            mode
        ][seed]
    )

    return max(
        row[
            "mean_gates"
        ]
        for row in logs
    )


candidate_models = []

for mode in TRAIN_MODES:
    for seed in TRAIN_SEEDS:
        candidate_models.append({
            "mode": mode,
            "seed": seed,
            "score": (
                best_validation_score(
                    mode,
                    seed
                )
            ),
            "path": (
                best_checkpoint_paths[
                    mode
                ][seed]
            ),
        })

candidate_models = sorted(
    candidate_models,
    key=lambda row: row[
        "score"
    ],
    reverse=True
)

print(
    f"{'Mode':12s} | "
    f"{'Seed':>4s} | "
    f"{'Best VAL gates':>14s}"
)

print("-" * 38)

for row in candidate_models:
    print(
        f"{row['mode']:12s} | "
        f"{row['seed']:4d} | "
        f"{row['score']:14.2f}"
    )


best_model_info = (
    candidate_models[0]
)

best_racing_agent = (
    SACAgent.load(
        best_model_info[
            "path"
        ],
        device=DEVICE,
    )
)

print()
print(
    "Selected best racing policy:"
)

print(
    best_model_info
)


## 16. Final unseen-track test — Direct vs Smooth


In [ ]:
final_test_results = {
    mode: {}
    for mode in TRAIN_MODES
}


def run_final_test(mode):
    """Valuta su TEST_TRACK_SEEDS (mai visti) il best checkpoint di
    ogni seed per il metodo `mode`, e salva i risultati in
    final_test_results[mode]."""

    final_test_results.setdefault(mode, {})

    print()
    print(
        "FINAL UNSEEN TEST:",
        mode
    )

    for seed in TRAIN_SEEDS:
        agent = SACAgent.load(
            best_checkpoint_paths[
                mode
            ][seed],
            device=DEVICE,
        )

        result = evaluate_on_tracks(
            agent,
            TEST_TRACK_SEEDS,
            episodes_per_track=1,
        )

        final_test_results[
            mode
        ][seed] = result

        print(
            f"seed {seed} | "
            f"gates {result['mean_gates']:.2f} | "
            f"success {result['success_rate']:.2f} | "
            f"crash {result['crash_rate']:.2f} | "
            f"miss {result['miss_rate']:.2f}"
        )


for mode in TRAIN_MODES:
    run_final_test(mode)


In [ ]:
def aggregate_test_metric(
    mode,
    key,
):
    values = np.array([
        final_test_results[
            mode
        ][seed][key]
        for seed
        in TRAIN_SEEDS
    ], dtype=float)

    return (
        float(
            np.mean(
                values
            )
        ),
        float(
            np.std(
                values
            )
        ),
    )


def print_final_test_table(modes):
    """Tabella finale mean +/- std sul test unseen, per uno o piu'
    metodi."""

    print(
        f"{'Method':14s} | "
        f"{'Gates':>12s} | "
        f"{'Success':>12s} | "
        f"{'Crash':>12s} | "
        f"{'Miss':>12s}"
    )

    print("-" * 72)

    for mode in modes:
        gates = aggregate_test_metric(mode, "mean_gates")
        success = aggregate_test_metric(mode, "success_rate")
        crash = aggregate_test_metric(mode, "crash_rate")
        miss = aggregate_test_metric(mode, "miss_rate")

        print(
            f"{mode:14s} | "
            f"{gates[0]:5.2f}\u00b1{gates[1]:4.2f} | "
            f"{success[0]:5.2f}\u00b1{success[1]:4.2f} | "
            f"{crash[0]:5.2f}\u00b1{crash[1]:4.2f} | "
            f"{miss[0]:5.2f}\u00b1{miss[1]:4.2f}"
        )


print_final_test_table(TRAIN_MODES)


## 17. Rollout su unseen track

In [ ]:
@torch.no_grad()
def rollout_agent(
    agent,
    track_seed,
):
    env = DroneGateRacing3DEnv(
        mode="evaluation",
        seed=track_seed
    )

    obs = env.reset(
        seed=track_seed,
        track_seed=track_seed,
        deterministic=True,
    )

    states = [
        env.state.copy()
    ]

    infos = []

    done = False

    while not done:
        action = agent.act(
            obs,
            deterministic=True
        )

        (
            obs,
            _,
            terminated,
            truncated,
            info
        ) = env.step(
            action
        )

        states.append(
            env.state.copy()
        )

        infos.append(
            info.copy()
        )

        done = bool(
            terminated
            or truncated
        )

    return (
        env,
        np.array(
            states
        ),
        infos,
    )


rollout_track_seed = (
    TEST_TRACK_SEEDS[0]
)

(
    rollout_env,
    rollout_states,
    rollout_infos
) = rollout_agent(
    best_racing_agent,
    rollout_track_seed,
)


fig = plt.figure(
    figsize=(11, 7)
)

ax = fig.add_subplot(
    111,
    projection="3d"
)

ax.plot(
    rollout_states[:, 0],
    rollout_states[:, 1],
    rollout_states[:, 2],
    linewidth=2,
    label="RL trajectory"
)

track = rollout_env.track

for i, center in enumerate(
    track["positions"]
):
    ax.scatter(
        [center[0]],
        [center[1]],
        [center[2]],
        marker="o"
    )

    ax.text(
        center[0],
        center[1],
        center[2] + 0.12,
        str(i + 1)
    )

ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_zlabel("z [m]")

ax.set_title(
    f"Best policy on unseen track seed {rollout_track_seed}"
)

ax.legend()

plt.show()

print(
    "Final info:",
    rollout_infos[-1]
)


## 18. Animazione 3D single-drone

In [ ]:
def compute_track_view_bounds(
    positions,
    xy_margin=0.8,
    z_margin=0.4,
):
    """Limiti degli assi calcolati dall'ingombro reale della pista
    (incluso il punto di partenza), invece di limiti fissi pensati
    per un percorso monotono lungo +x. Riusata da tutti i grafici e
    le animazioni 3D cosi' funzionano anche con piste che virano o
    tornano indietro."""

    xy = positions[:, :2]

    x_min = (
        min(0.0, float(xy[:, 0].min()))
        - xy_margin
    )

    x_max = (
        max(0.0, float(xy[:, 0].max()))
        + xy_margin
    )

    y_min = (
        min(0.0, float(xy[:, 1].min()))
        - xy_margin
    )

    y_max = (
        max(0.0, float(xy[:, 1].max()))
        + xy_margin
    )

    z_min = max(
        0.0,
        float(positions[:, 2].min())
        - z_margin,
    )

    z_max = (
        float(positions[:, 2].max())
        + z_margin
    )

    return (
        (x_min, x_max),
        (y_min, y_max),
        (z_min, z_max),
    )


def draw_gate_3d(
    ax,
    center,
    normal,
    radius=0.60,
    label=None,
):
    center = np.asarray(
        center,
        dtype=np.float32
    )

    normal = normalize_vector(
        normal
    )

    # Costruisco una base ortogonale
    # appartenente al piano del gate.
    up = np.array(
        [0.0, 0.0, 1.0],
        dtype=np.float32
    )

    side = np.cross(
        normal,
        up
    )

    if np.linalg.norm(side) < 1e-4:
        side = np.array(
            [0.0, 1.0, 0.0],
            dtype=np.float32
        )

    side = normalize_vector(
        side
    )

    vertical = normalize_vector(
        np.cross(
            side,
            normal
        )
    )

    theta = np.linspace(
        0.0,
        2.0 * np.pi,
        100
    )

    ring = (
        center[None, :]
        + radius
        * np.cos(theta)[:, None]
        * side[None, :]
        + radius
        * np.sin(theta)[:, None]
        * vertical[None, :]
    )

    # Ring del gate
    ax.plot(
        ring[:, 0],
        ring[:, 1],
        ring[:, 2],
        linewidth=3.0,
        alpha=0.95
    )

    # Centro del gate
    ax.scatter(
        [center[0]],
        [center[1]],
        [center[2]],
        s=35
    )

    # Normale del gate
    ax.quiver(
        center[0],
        center[1],
        center[2],
        normal[0],
        normal[1],
        normal[2],
        length=0.35,
        normalize=True
    )

    if label is not None:
        ax.text(
            center[0],
            center[1],
            center[2] + radius + 0.10,
            str(label),
            fontsize=11,
            fontweight="bold"
        )


def animate_single_drone(
    env,
    states,
    interval=45,
    frame_skip=2,
):
    frames = np.arange(
        0,
        len(states),
        frame_skip
    )

    fig = plt.figure(
        figsize=(12, 8)
    )

    ax = fig.add_subplot(
        111,
        projection="3d"
    )

    positions = env.track[
        "positions"
    ]

    normals = env.track[
        "normals"
    ]

    # --------------------------------------------------
    # LIMITI CAMERA
    # --------------------------------------------------

    xlim, ylim, zlim = (
        compute_track_view_bounds(
            positions
        )
    )

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")

    ax.set_title(
        "3D Drone Racing — Unseen Track"
    )

    # Vista leggermente dall'alto
    ax.view_init(
        elev=24,
        azim=-65
    )

    # --------------------------------------------------
    # DISEGNA I GATE
    # --------------------------------------------------

    for i, (
        center,
        normal
    ) in enumerate(
        zip(
            positions,
            normals
        )
    ):
        draw_gate_3d(
            ax,
            center=center,
            normal=normal,
            radius=0.60,
            label=i + 1
        )

    # Linea ideale tra i gate
    ax.plot(
        positions[:, 0],
        positions[:, 1],
        positions[:, 2],
        linestyle="--",
        linewidth=1.0,
        alpha=0.45,
        label="Track centerline"
    )

    # --------------------------------------------------
    # DRONE + TRAIETTORIA
    # --------------------------------------------------

    trajectory, = ax.plot(
        [],
        [],
        [],
        linewidth=2.5,
        label="RL trajectory"
    )

    drone_point, = ax.plot(
        [],
        [],
        [],
        marker="o",
        markersize=10,
        label="Drone"
    )

    ax.legend(
        loc="upper left"
    )

    def update(
        frame_number
    ):
        i = frames[
            frame_number
        ]

        history = states[
            :i + 1
        ]

        trajectory.set_data(
            history[:, 0],
            history[:, 1]
        )

        trajectory.set_3d_properties(
            history[:, 2]
        )

        point = states[i]

        drone_point.set_data(
            [point[0]],
            [point[1]]
        )

        drone_point.set_3d_properties(
            [point[2]]
        )

        return (
            trajectory,
            drone_point
        )

    animation = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval,
        blit=False,
    )

    plt.close(fig)

    return animation


single_animation = (
    animate_single_drone(
        rollout_env,
        rollout_states
    )
)

display(
    HTML(
        single_animation.to_jshtml()
    )
)


## 19. Direct vs Smooth — final comparison

The key result is whether Smooth reaches the same validation performance with fewer environment transitions and whether that advantage transfers to unseen tracks.


In [ ]:
print("DIRECT vs SMOOTH — FINAL UNSEEN TEST")
print_final_test_table(
    [
        "direct",
        "smooth",
    ]
)

print()
print("FIRST VALIDATION STEP REACHING SUCCESS >= 0.60")

for mode in [
    "direct",
    "smooth",
]:
    first_steps = []

    for seed in TRAIN_SEEDS:
        hits = [
            row["step"]
            for row
            in all_validation_logs[
                mode
            ][seed]
            if row["success_rate"] >= 0.60
        ]

        first_steps.append(
            min(hits)
            if hits
            else np.nan
        )

    arr = np.array(
        first_steps,
        dtype=float
    )

    print(
        f"{mode:8s}: "
        f"{np.nanmean(arr):.0f} ± "
        f"{np.nanstd(arr):.0f} transitions "
        f"| per-seed {first_steps}"
    )

print()
print(
    "Interpretation: if Smooth reaches the same validation success "
    "earlier than Direct under the same budget and final-task validation, "
    "it demonstrates improved sample efficiency."
)
